This is the notebook used for fine-tuning, as the main objective of Bachelor's Thesis.

*"Evaluation and Implementation of a Generative AI-Based Wellness Assistant"*

*Ulvi Aliyev*

The next command will show currently connect GPU for debugging purposes.

In [ ]:
!nvidia-smi

In the next cell, dependencies will be installed.

In [ ]:
!pip install -q -U bitsandbytes>=0.46.1
!pip install -q datasets
!pip install -q -U git+https://github.com/huggingface/transformers.git
!pip install -q -U git+https://github.com/huggingface/peft.git
!pip install -q -U git+https://github.com/huggingface/accelerate.git

In the next cell, the mode parameter will be initialized for fine tuning, as I conducted fine-tuning with several test subsets before continuing with the full dataset.

In [ ]:
MODE = "full"

run_cfg = {
    "subset": {
        "train_size":  5000,
        "val_size":    500,
        "epochs":      1,
        "output_dir":  "./health-coach-subset",
        "drive_dir":   "/content/drive/MyDrive/Fine-tuning_Thesis/OutputModel",
    },
    "full": {
        "train_size":  None,
        "val_size":    None,
        "epochs":      3,
        "output_dir":  "./health-coach-full",
        "drive_dir":   "/content/drive/MyDrive/Fine-tuning_Thesis/OutputModel_Full",
    },
}[MODE]

print(f"Mode      : {MODE}")
print(f"Epochs    : {run_cfg['epochs']}")
print(f"Train size: {run_cfg['train_size'] or 'full'}")
print(f"Val size  : {run_cfg['val_size'] or 'full'}")

After installing required dependencies, the model gets imported and initialized.

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Change to 7B
model_id = "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B"

# QLoRA config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

# Load model
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = tokenizer.eos_token_id

Pre-process model to prepare for training.

In [ ]:
from peft import prepare_model_for_kbit_training

model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

Get trainable parameters for better debugging.

In [ ]:
from peft import LoraConfig, get_peft_model

config = LoraConfig(
    r=16,
    lora_alpha=32,
    # Target modules for Qwen architecture
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, config)
model.print_trainable_parameters()  # Built-in function


In the next chunk, the dataset used for training will be loaded. Since `train.jsonl` and `val.jsonl` files are in Google Drive, it should be mounted beforehand. Next, the dataset will be formatted in "Alpaca" style for instruction tuning.

In [ ]:
from datasets import load_dataset
from google.colab import drive

# Mounting Google Drive
drive.mount('/content/drive')

# Load dataset
dataset = load_dataset(
    "json",
    data_files={
        "train": "/content/drive/MyDrive/Fine-tuning_Thesis/train.jsonl",
        "validation": "/content/drive/MyDrive/Fine-tuning_Thesis/val.jsonl"
    }
)

print(f"Loaded {len(dataset['train']):,} training samples")
print(f"Loaded {len(dataset['validation']):,} validation samples")

# Format function
def format_instruction(sample):
    instruction = sample["instruction"]
    input_text = sample["input"]
    output = sample["output"]

    if input_text.strip():
        prompt = f"""Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{instruction}

### Input:
{input_text}

### Response:
{output}"""
    else:
        prompt = f"""Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
{instruction}

### Response:
{output}"""

    return {"text": prompt}

# Apply formatting
dataset = dataset.map(format_instruction)

print("\nSample prompt:")
print("="*60)
print(dataset["train"][0]["text"][:500] + "...")

In [ ]:
train_ds = dataset["train"].shuffle(seed=42)
val_ds   = dataset["validation"].shuffle(seed=42)

if run_cfg["train_size"]:
    train_ds = train_ds.select(range(run_cfg["train_size"]))
if run_cfg["val_size"]:
    val_ds = val_ds.select(range(run_cfg["val_size"]))

test_train = train_ds
test_val   = val_ds

print(f"Train: {len(test_train):,} samples")
print(f"Val  : {len(test_val):,} samples")

In the next cell, the training itself will be happening.

In [13]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=512,
        padding=False,
    )

print("Tokenizing dataset...")
tokenized_train = test_train.map(tokenize_function, batched=True, remove_columns=test_train.column_names)
tokenized_val   = test_val.map(tokenize_function, batched=True, remove_columns=test_val.column_names)

print(f"Tokenized {len(tokenized_train):,} training samples")
print(f"Tokenized {len(tokenized_val):,} validation samples")

training_args = TrainingArguments(
    output_dir=run_cfg["output_dir"],
    num_train_epochs=run_cfg["epochs"],
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_steps=50,
    weight_decay=0.01,
    bf16=True,
    fp16=False,
    optim="adamw_torch_fused",
    gradient_checkpointing=True,
    dataloader_num_workers=4,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
    pad_to_multiple_of=8,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
)

model.config.use_cache = False

print("="*60)
print(f"STARTING TRAINING  [{MODE.upper()}]")
print("="*60)
print(f"Model  : {model_id}")
print(f"Train  : {len(tokenized_train):,} samples")
print(f"Val    : {len(tokenized_val):,} samples")
print(f"Epochs : {run_cfg['epochs']}")
print("="*60 + "\n")

trainer.train()
print("\n✓ Training complete!")

Tokenizing dataset...
Tokenized 195,229 training samples
Tokenized 21,693 validation samples
STARTING TRAINING  [FULL]
Model  : deepseek-ai/DeepSeek-R1-Distill-Qwen-7B
Train  : 195,229 samples
Val    : 21,693 samples
Epochs : 3



Epoch,Training Loss,Validation Loss
1,0.610735,0.593603
2,0.470685,0.496069
3,0.398604,0.452598



✓ Training complete!


In the next cell, model adapter will be saved.

In [14]:
trainer.save_model(run_cfg["output_dir"])
tokenizer.save_pretrained(run_cfg["output_dir"])

print(f"Model saved to {run_cfg['output_dir']}")

Model saved to ./health-coach-full


Export to Google Drive

In [15]:
import shutil

shutil.copytree(run_cfg["output_dir"], run_cfg["drive_dir"], dirs_exist_ok=True)
print(f"✓ Exported to {run_cfg['drive_dir']}")

✓ Exported to /content/drive/MyDrive/Fine-tuning_Thesis/OutputModel_Full
